In [1]:
# Import modules.
# ---------------------------------------------------------
import pyvisa
import struct
import math
import time 
import numpy as np

class siglent_scope():
    def __init__(self,identifier_pattern):
        self.sds = None
        rm = pyvisa.ResourceManager()
        resource_list = rm.list_resources()
        for r in resource_list:
            if identifier_pattern in r:
                self.sds = rm.open_resource(r)
                self.resource_id = r
                print("Found SDS scope:", self.sds.query("*IDN?"))
                break
        if not self.sds:
            print("Failed to find scope based on identifier pattern '%s'" % identifier_pattern)
            print("Currently availabe VISA instruments are:")
            for r in resource_list:
                print(r)
        # Visa settings we need:
        self.sds.timeout = 2000 # default value is 2000(2s)
        self.sds.chunk_size = 20 * 1024 * 1024 # default value is 20*1024(20k bytes)
        self.HORI_NUM = 10
        self.tdiv_enum = [200e-12,500e-12, 1e-9,\
         2e-9, 5e-9, 10e-9, 20e-9, 50e-9, 100e-9, 200e-9, 500e-9, \
         1e-6, 2e-6, 5e-6, 10e-6, 20e-6, 50e-6, 100e-6, 200e-6, 500e-6, \
         1e-3, 2e-3, 5e-3, 10e-3, 20e-3, 50e-3, 100e-3, 200e-3, 500e-3, \
         1, 2, 5, 10, 20, 50, 100, 200, 500, 1000]

    # Most of this is still copy-pasted from the programming manual. Not super 
    # transparent code, but it does work. 
    def parse_preamble(self, recv):
        WAVE_ARRAY_1 = recv[0x3c:0x3f + 1]
        wave_array_count = recv[0x74:0x77 + 1]
        first_point = recv[0x84:0x87 + 1]
        sp = recv[0x88:0x8b + 1]
        v_scale = recv[0x9c:0x9f + 1]
        v_offset = recv[0xa0:0xa3 + 1]
        interval = recv[0xb0:0xb3 + 1]
        code_per_div = recv[0xa4:0Xa7 + 1]
        adc_bit = recv[0xac:0Xad + 1]
        delay = recv[0xb4:0xbb + 1]
        tdiv = recv[0x144:0x145 + 1]
        probe = recv[0x148:0x14b + 1]
        data_bytes = struct.unpack('i', WAVE_ARRAY_1)[0]
        point_num = struct.unpack('i', wave_array_count)[0]
        fp = struct.unpack('i', first_point)[0]
        sp = struct.unpack('i', sp)[0]
        interval = struct.unpack('f', interval)[0]
        delay = struct.unpack('d', delay)[0]
        tdiv_index = struct.unpack('h', tdiv)[0]
        probe = struct.unpack('f', probe)[0]
        vdiv = struct.unpack('f', v_scale)[0] * probe
        offset = struct.unpack('f', v_offset)[0] * probe
        code = struct.unpack('f', code_per_div)[0]
        adc_bit = struct.unpack('h', adc_bit)[0]
        tdiv = self.tdiv_enum[tdiv_index]
        return vdiv, offset, interval, delay, tdiv, code, adc_bit

    # Same here: mostly ugly code copy-pasted from user manual
    def get_waveform(self, channel_number, npoints = None):
        sds = self.sds # ugly but oh well
        # Get the channel waveform parameter data blocks and parse them
        if channel_number > 0 and channel_number < 5:
            # First check if channel is on: if not, then just return
            if "OFF" in sds.query("CHAN%d:SWIT?" % channel_number):
                print("Warning: Channel %d is currently off, please turn it on first" % channel_number)
                return
            else:
                sds.write("WAV:SOUR C%d" % channel_number)
        else:
            print("Channel number must be a number from 1 to 4")
            return
        # Get the preamble
        sds.write("WAV:PREamble?")
        recv_all = sds.read_raw()
        # Find the starting byte 
        recv = recv_all[recv_all.find(b'#') + 11:]
        # Parse the preamble
        vdiv, ofst, interval, trdl, tdiv, vcode_per, adc_bit = self.parse_preamble(recv)
        # Set the starting datapoint for the transfer
        sds.write(":WAVeform:STARt 0")
        # Get the waveform points and confirm the number of waveform slice reads
        points_in_trace = float(sds.query(":ACQuire:POINts?").strip())
        if not npoints:
            points = points_in_trace
        else:
            if npoints > points_in_trace:
                print("Warning: trace contains only %e points while %e requested" 
                      % (points_in_trace, npoints))
        one_piece_num = float(sds.query(":WAVeform:MAXPoint?").strip())
        read_times = math.ceil(points / one_piece_num)
        # Set the number of read points per slice, if the waveform points is greater than the maximum
        # number of slice reads
        if points > one_piece_num:
            sds.write(":WAVeform:POINt {}".format(one_piece_num))
            # Choose the format of the data returned
            sds.write(":WAVeform:WIDTh BYTE")
        if adc_bit > 8:
            sds.write(":WAVeform:WIDTh WORD")
        #Get the waveform data for each slice
        recv_byte = b'' # initiate variable as a binary string
        for i in range(0, read_times):
            start = i * one_piece_num
            #Set the starting point of each slice
            sds.write(":WAVeform:STARt {}".format(start))
            #Get the waveform data of each slice
            t0 = time.time()
            sds.write("WAV:DATA?")
            recv_rtn = sds.read_raw()
            # print("read time block %d bytes_read %d time %.2f seconds" % (i, len(recv_rtn), (time.time()-t0)))
            # We need to get rid of the two "\n\n" at the end of the received bytes
            # However, we should not use rstrip since it's behavior is not so well defined for 
            # binary streams. But fortunately, there are always two, so we can just get rid of them
            # using slicing. 
            #recv_rtn = recv_rtn.rstrip()
            recv_rtn = recv_rtn[:-2]
            #Splice each waveform data based on data block information
            block_start = recv_rtn.find(b'#')
            data_digit = int(recv_rtn[block_start + 1:block_start + 2])
            data_start = block_start + 2 + data_digit
            bytes_received = len(recv_rtn[data_start:])
            bytes_expected =  (points % one_piece_num) * 2 # hack, the SDS814x is always 16 bit
            if bytes_received < bytes_expected:
                print("Warning: possible short read on data tranfer block", i)
                print("Bytes expected %d, Bytes received %d" % (bytes_expected, bytes_received))
            recv_byte += recv_rtn[data_start:]
        # Unpack signed byte data.
        if adc_bit > 8:
            #print("points", points)
            #print("len(recv_byte)", len(recv_byte))
            convert_data = struct.unpack("=%dh"%points, recv_byte)
        else:
            convert_data = struct.unpack("%db"%points, recv_byte)
        convert_data = np.array(convert_data)
        #Calculate the voltage value and time value
        time_value = []
        volt_value = []
        N = len(convert_data)
        i = np.linspace(0, N-1, N)
        volt_value = convert_data / vcode_per * float(vdiv) -float(ofst)
        time_data = float(tdiv)*self.HORI_NUM/2 + i*interval + float(trdl)
        # The scope fails to transfer data if we ask for data too quickly :(
        # Trail and error suggests we need a 50 ms waiting time here
        # Kindof crappy, but alright considering that any read always
        # takes 200 ms. 
        time.sleep(0.05) 
        return time_data, volt_value
        

In [2]:
scope = siglent_scope("SDS")

Found SDS scope: Siglent Technologies,SDS814X HD,SDS08A0Q808362,3.8.12.1.1.3.8



In [3]:
if "OFF" in scope.sds.query("CHAN3:SWIT?"):
    print("Channel is off")

Channel is off


In [4]:
scope.get_waveform(1)
scope.get_waveform(2)

(array([0.0017    , 0.00170002, 0.00170004, ..., 0.00189994, 0.00189996,
        0.00189998], shape=(10000,)),
 array([0.00416667, 0.004375  , 0.00395833, ..., 0.004375  , 0.00479167,
        0.004375  ], shape=(10000,)))

In [12]:
import bokeh
from bokeh.models import ColumnDataSource
from bokeh.plotting import figure, show
from bokeh.io import output_notebook, push_notebook
output_notebook()
import ipywidgets as widgets
from IPython.display import display

import nest_asyncio
nest_asyncio.apply()
import IPython
do_one_iteration = IPython.get_ipython().kernel.do_one_iteration

Loading BokehJS ...

In [ ]:
t,v = scope.get_waveform(2)
x = t-t[0]
y = v
p = figure(height=200, width=600) 
p.sizing_mode = "scale_width"
#l=p.dot(x,y)
l=p.line(x,y)
target = show(p, notebook_handle=True)
stop_button = widgets.ToggleButton()
stop_button.description = "Stop"
display(stop_button)

while True:
    t,v = scope.get_waveform(1)
    l.data_source.data = dict(x=t, y=v)
    push_notebook(handle=target)
    # Process updates from ipywdiget GUI objects
    await do_one_iteration()
    if stop_button.value:
        break

ToggleButton(value=False, description='Stop')

In [19]:
print("foo")

foo
